# This notebook represent multimodel training 

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

### 1. Load data set

In [8]:
# Load scaled features and target
X_train_scaled = np.load("../data/processed/X_train_scaled.npy")
X_test_scaled  = np.load("../data/processed/X_test_scaled.npy")
y_train_scaled = np.load("../data/processed/y_train_scaled.npy")
y_test_scaled  = np.load("../data/processed/y_test_scaled.npy")

# Load original target for evaluation
y_train = np.load("../data/processed/y_train.npy")
y_test  = np.load("../data/processed/y_test.npy")

# Load scalers
scaler_X = joblib.load("../artifacts/scalers/scaler_X.pkl")
scaler_y = joblib.load("../artifacts/scalers/scaler_y.pkl")

# Load selected feature names
selected_features = joblib.load("../artifacts/scalers/selected_features.pkl")

print("Data loaded successfully!")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape :", X_test_scaled.shape)
print("y_train shape       :", y_train.shape)
print("y_test shape        :", y_test.shape)
print("Number of features  :", len(selected_features))

Data loaded successfully!
X_train_scaled shape: (15759, 24)
X_test_scaled shape : (3940, 24)
y_train shape       : (15759,)
y_test shape        : (3940,)
Number of features  : 24


### 2. Define multi model

In [10]:
models = {
            "Linear Regression": LinearRegression(),
            "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
            "XGBoost": XGBRegressor(n_estimators=100, random_state=42)
            }

### 3. Train models

In [12]:
trained_models = {}

print("Starting Model Training...\n")

for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train_scaled, y_train)
    trained_models[model_name] = model
    print(f"{model_name} trained successfully!\n")

print("All models trained successfully!")

Starting Model Training...

Training Linear Regression...
Linear Regression trained successfully!

Training Random Forest...
Random Forest trained successfully!

Training XGBoost...
XGBoost trained successfully!

All models trained successfully!


### 4. Evaluation models

In [14]:
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"{model_name} Results:")
    print(f"  MAE  : {mae:.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  R²   : {r2:.4f}\n")
    
    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [15]:
results = []

print("Starting Model Evaluation...\n")

for model_name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    result = evaluate_model(y_test, y_pred, model_name)
    results.append(result)

Starting Model Evaluation...

Linear Regression Results:
  MAE  : 26.4381
  RMSE : 59.7238
  R²   : 0.5644

Random Forest Results:
  MAE  : 53.3231
  RMSE : 87.3507
  R²   : 0.0681

XGBoost Results:
  MAE  : 113.3462
  RMSE : 155.0843
  R²   : -1.9375



### 5. Save best model

In [17]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="MAE").reset_index(drop=True)

print("Baseline Models Performance Comparison:")
display(results_df)

Baseline Models Performance Comparison:


,Model,MAE,RMSE,R2
0,Linear Regression,26.438120,59.723763,0.564356
1,Random Forest,53.323096,87.350708,0.068099
2,XGBoost,113.346222,155.084270,-1.937464


In [18]:
os.makedirs("../artifacts/models", exist_ok=True)

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

save_path = f"../artifacts/models/{best_model_name.replace(' ', '_').lower()}_model.pkl"
joblib.dump(best_model, save_path)

print(f"Best model '{best_model_name}' saved at: {save_path}")

Best model 'Linear Regression' saved at: ../artifacts/models/linear_regression_model.pkl
